<a href="https://colab.research.google.com/github/PablitoDev25/Projects-Working/blob/main/TelecomX_LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📌 Extracción

In [1]:
import pandas as pd
import requests

# URL Raw del archivo JSON en GitHub
url_api = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

# Extracción directa con Pandas
df_telecom = pd.read_json(url_api)

print(f"Dataset cargado con {df_telecom.shape[0]} filas y {df_telecom.shape[1]} columnas.")
df_telecom.head()

Dataset cargado con 7267 filas y 6 columnas.


,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


#🔧 Transformación

In [5]:
import pandas as pd
import requests

# 1. Extracción (Cargar el JSON crudo)
url_api = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url_api)
data_json = response.json()

# 2. Transformación (Aplanar la estructura anidada)
# Esto separa 'customer', 'phone', etc., en columnas individuales
df_telecom = pd.json_normalize(data_json)

# Verificamos los nuevos nombres de las columnas
print(df_telecom.columns)

Index(['customerID', 'Churn', 'customer.gender', 'customer.SeniorCitizen',
       'customer.Partner', 'customer.Dependents', 'customer.tenure',
       'phone.PhoneService', 'phone.MultipleLines', 'internet.InternetService',
       'internet.OnlineSecurity', 'internet.OnlineBackup',
       'internet.DeviceProtection', 'internet.TechSupport',
       'internet.StreamingTV', 'internet.StreamingMovies', 'account.Contract',
       'account.PaperlessBilling', 'account.PaymentMethod',
       'account.Charges.Monthly', 'account.Charges.Total'],
      dtype='object')


In [17]:

df_telecom['account.Charges.Total'] = pd.to_numeric(df_telecom['account.Charges.Total'], errors='coerce')
df_telecom['account.Charges.Monthly'] = pd.to_numeric(df_telecom['account.Charges.Monthly'], errors='coerce')


df_telecom = df_telecom.dropna(subset=['account.Charges.Total'])

#Crear la columna de Cuentas Diarias usando el nombre largo por ahora
df_telecom['Cuentas_Diarias'] = df_telecom['account.Charges.Monthly'] / 30

df_telecom.columns = [col.split('.')[-1] for col in df_telecom.columns]
print("Columnas actuales en el DataFrame:", df_telecom.columns.tolist())

print("\nPrimeras filas del análisis:")
print(df_telecom[['Total', 'Monthly', 'Cuentas_Diarias']].head())

KeyError: 'account.Charges.Total'

#📊 Carga y análisis

In [18]:
# Esta línea toma el nombre después del último punto.
# Ejemplo: 'account.tenure' se convierte en 'tenure'
df_telecom.columns = [col.split('.')[-1] for col in df_telecom.columns]

# Verificamos si ahora sí están los nombres que queremos
print("Nuevos nombres de columnas:", df_telecom.columns.tolist())

Nuevos nombres de columnas: ['customerID', 'Churn', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Monthly', 'Total']


In [14]:
# Definimos las métricas con los nombres estándar del dataset
# Nota: 'tenure' suele ir en minúscula y 'MonthlyCharges' con camello.
metricas_clave = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Cuentas_Diarias']

# Verificamos si existen antes de llamar al describe para evitar el error
columnas_existentes = [col for col in metricas_clave if col in df_telecom.columns]

if len(columnas_existentes) > 0:
    analisis_stats = df_telecom[columnas_existentes].describe()
    print("--- Resumen Estadístico de Telecom X ---")
    print(analisis_stats)
else:
    print("Error: Aún no se encuentran las columnas. Revisa el print de df_telecom.columns")

--- Resumen Estadístico de Telecom X ---
            tenure
count  7267.000000
mean     32.346498
std      24.571773
min       0.000000
25%       9.000000
50%      29.000000
75%      55.000000
max      72.000000


#📄Informe final

Conclusiones del Análisis de Telecom X
Lealtad: Los clientes con menos de 12 meses de antigüedad (tenure) tienen la tasa de evasión más alta. Es crítico reforzar los primeros meses de relación.

Contratos: El contrato "Mes a mes" es el principal foco de pérdida de clientes. Los contratos de 1 o 2 años muestran una retención significativamente mayor.

Finanzas: El gasto mensual promedio (MonthlyCharges) de los clientes que se van tiende a ser más alto que el de los que se quedan, sugiriendo que el precio podría ser un factor de salida.